dataset link "https://www.kaggle.com/datasets/praveengovi/emotions-dataset-for-nlp?datasetId=605165&sortBy=voteCount"

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import re
import nltk
from nltk.corpus import stopwords

In [3]:
# Read the training data from file
train_df = pd.read_csv("train.txt", header=None, sep=";", names=["Comment", "Emotion"], encoding="utf-8")

# Read the test data from file
test_df = pd.read_csv("test.txt", header=None, sep=";", names=["Comment", "Emotion"], encoding="utf-8")

# Read the validation data from file
val_df = pd.read_csv("val.txt", header=None, sep=";", names=["Comment", "Emotion"], encoding="utf-8")


In [4]:
# Combine the training and validation data
train_val_df = pd.concat([train_df, val_df], ignore_index=True)



In [5]:
# Split the data into features and labels
train_x = train_val_df["Comment"]
train_y = train_val_df["Emotion"]
test_x = test_df["Comment"]
test_y = test_df["Emotion"]


In [6]:
# Define a preprocessing function to remove HTML tags, tokenize, remove stopwords, and convert to lowercase
nltk.download("punkt")
nltk.download("stopwords")
stop = set(stopwords.words("english"))

import re
def striphtml(data):
    p = re.compile(r'<.*?>')
    return p.sub('', data)

def remove_stopwords(text):
    filtered_words = " ".join([word.lower() for word in text.split() if word.lower() not in stop])
    filtered_words = striphtml(filtered_words)
    return filtered_words


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Mickey\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mickey\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# Apply preprocessing to train and test data
train_x = train_x.apply(remove_stopwords)
test_x = test_x.apply(remove_stopwords)


In [8]:
# Vectorize the data
vectorizer = TfidfVectorizer()
train_x_vec = vectorizer.fit_transform(train_x)
test_x_vec = vectorizer.transform(test_x)

In [9]:
# Train the model
model = MultinomialNB()
model.fit(train_x_vec, train_y)


MultinomialNB()

In [10]:
# Evaluate the model on the test data
test_y_pred = model.predict(test_x_vec)
test_accuracy = accuracy_score(test_y, test_y_pred)
test_confusion_matrix = confusion_matrix(test_y, test_y_pred)
test_classification_report = classification_report(test_y, test_y_pred)

C:\Users\Mickey\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Mickey\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Mickey\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1

In [11]:
print(f"Test Accuracy: {test_accuracy}")
print(f"Test Confusion Matrix:\n{test_confusion_matrix}")
print(f"Test Classification Report:\n{test_classification_report}")

Test Accuracy: 0.6915
Test Confusion Matrix:
[[ 93   1  93   0  88   0]
 [  4  66  73   0  81   0]
 [  0   0 687   0   8   0]
 [  1   0 114   8  36   0]
 [  0   1  51   0 529   0]
 [  0   3  40   0  23   0]]
Test Classification Report:
              precision    recall  f1-score   support

       anger       0.95      0.34      0.50       275
        fear       0.93      0.29      0.45       224
         joy       0.65      0.99      0.78       695
        love       1.00      0.05      0.10       159
     sadness       0.69      0.91      0.79       581
    surprise       0.00      0.00      0.00        66

    accuracy                           0.69      2000
   macro avg       0.70      0.43      0.44      2000
weighted avg       0.74      0.69      0.63      2000



In [12]:
# New text data for prediction
new_text = [
    "I feel happy today",
    "This movie is boring.",
    "I think it's the easiest time of year to feel dissatisfied.",
    "I am just feeling cranky and blue."
]

In [13]:
# Preprocess the new text data
new_text_preprocessed = pd.Series(new_text).apply(remove_stopwords)

# Vectorize the new text data
new_text_vectorized = vectorizer.transform(new_text_preprocessed)


In [14]:
# Make predictions using the trained model
predictions = model.predict(new_text_vectorized)

In [15]:
# Print the predictions
for text, prediction in zip(new_text, predictions):
    print(f"Text: {text}")
    print(f"Predicted Emotion: {prediction}")
    print()

Text: I feel happy today
Predicted Emotion: joy

Text: This movie is boring.
Predicted Emotion: sadness

Text: I think it's the easiest time of year to feel dissatisfied.
Predicted Emotion: anger

Text: I am just feeling cranky and blue.
Predicted Emotion: anger

